In [1]:

import pandas as pd
import plotly.express as px


In [37]:

# Top countries with most rail freight?

import pandas as pd
import plotly.express as px

# Loading the dataset
df = pd.read_csv("../Processed/merged_eurostat_clean_V2.csv")

# Renaming columns for clarity
df = df.rename(columns={
    "geo": "country",
    "TIME_PERIOD": "year",
    "NST_TOTAL_MIO_TKM": "million_tkm"
})

# Converting freight column to numeric
df["million_tkm"] = pd.to_numeric(df["million_tkm"], errors="coerce")

# Summing total rail freight by country over all years
total_freight = df.groupby("country")["million_tkm"].sum().reset_index()

# Sorting and selecting Top 10
top10 = total_freight.sort_values(by="million_tkm", ascending=False).head(10)

# Plotting using Plotly
fig = px.bar(
    top10,
    x="country",
    y="million_tkm",
    labels={"country": "Country", "million_tkm": "Total Rail Freight (Million tonne-km)"},
    text="million_tkm",
    color="million_tkm",
    color_continuous_scale="Blues"
)

fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')

fig.update_layout(
    xaxis_tickangle=0,
    plot_bgcolor="white",
    font=dict(size=12),
    showlegend=False,
    coloraxis_colorbar_title="Freight Volume"
)

fig.add_annotation(
    text="<b>Figure 4.1 – Top 10 EU Countries by Total Rail Freight (Million tonne-km)</b>",
    xref="paper", yref="paper",
    x=0.2, y=-0.23,
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="left"
)

fig.show()




In [36]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Top 5
top5 = total_freight.sort_values(by="million_tkm", ascending=False).head(5)

# Bottom 5
bottom5 = (
    total_freight.sort_values(by="million_tkm", ascending=True)
    .head(5)
    .sort_values(by="million_tkm", ascending=False)
)

# Subplots
fig = make_subplots(
    rows=1, cols=2
)

# Top 5 Bar Chart
fig.add_trace(
    go.Bar(
        x=top5["country"],
        y=top5["million_tkm"],
        marker_color="steelblue"
    ),
    row=1, col=1
)

# Bottom 5 Bar Chart
fig.add_trace(
    go.Bar(
        x=bottom5["country"],
        y=bottom5["million_tkm"],
        marker_color="salmon"
    ),
    row=1, col=2
)

# Layout Formatting
fig.update_layout(
    title="Top vs Bottom 5 EU Countries by Total Rail Freight (Million tkm)",
    plot_bgcolor="white",
    showlegend=False,
    height=500
)


fig.update_xaxes(title_text="Country", tickangle=0, row=1, col=1)
fig.update_xaxes(title_text="Country", tickangle=0, row=1, col=2)

fig.update_yaxes(title_text="Total Rail Freight (Million tkm)", row=1, col=1)

fig.add_annotation(
    text="<b>Figure 4.2 (a) – Top 5 EU Countries by Total Rail Freight </b>",
    xref="paper", yref="paper",
    x=0, y=-0.23,
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="left"
)

fig.add_annotation(
    text="<b>Figure 4.2 (b) – Bottom 5 EU Countries by Total Rail Freight </b>",
    xref="paper", yref="paper",
    x=1, y=-0.23,
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="left"
)

fig.show()



In [42]:
# Analyzing freight trends over time for the top 5 countries
top5 = total_freight.sort_values(by="million_tkm", ascending=False).head(5)["country"].tolist()
df_top5 = df[df["country"].isin(top5)]
df_top5_yearly = df_top5.groupby(["year","country"])["million_tkm"].sum().reset_index()

# Plotly line chart
fig = px.line(
    df_top5_yearly,
    x="year",
    y="million_tkm",
    color="country",
    markers=True
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Rail Freight (million tkm)",
    plot_bgcolor="white",
    legend_title="Country",
    font=dict(size=12),
)

fig.update_xaxes(showgrid=True, gridwidth=0.3, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridwidth=0.3, gridcolor="lightgrey")

fig.add_annotation(
    text="<b>Figure 4.3 – Freight Trend Over Time for Top 5 EU Countries </b>",
    xref="paper", yref="paper",
    x=0.25, y=-0.23,
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="left"
)

fig.show()


In [44]:
from scipy.stats import chisquare
# Use total_freight to build the country-value series already computed earlier
total_sorted = total_freight.set_index("country")["million_tkm"].sort_values(ascending=False)

# Observed,expected values
observed = total_sorted.values
expected = [observed.mean()] * len(observed)

# Chi-square test
chi_stat, p_value = chisquare(f_obs=observed, f_exp=expected)

print("Chi-square Statistic:", chi_stat)
print("p-value:", p_value)

# Plotly bar chart
fig = go.Figure()

# Observed freight bars
fig.add_trace(go.Bar(
    x=total_sorted.index,
    y=total_sorted.values,
    marker_color="steelblue"
))

# Expected equal-share horizontal line
fig.add_hline(
    y=observed.mean(),
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text="Expected Value",
    annotation_position="top left"
)

# Layout
fig.update_layout(
    xaxis_title="Country",
    yaxis_title="Total Rail Freight (million tkm)",
    xaxis_tickangle=0,
    plot_bgcolor="white",
    showlegend=False,
    height=500
)

fig.add_annotation(
    text="<b>Figure 4.4 – Distribution of Rail Freight Across EU Countries </b>",
    xref="paper", yref="paper",
    x=0.25, y=-0.23,
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="left"
)
fig.show()


Chi-square Statistic: 15980911.611509945
p-value: 0.0
